# Data pipeline

How MedSegBench data is pulled, configured, and turned into batches. It's all driven by one `DataConfig`... a small YAML file in `configs/data/`

In [ ]:
import sys
sys.path.insert(0, "../src")

from medseg.data.medseg_data import (
    DataConfig, build_dataloaders, list_datasets, resolve_dataset_class,
)

## 1. Load a config

Each dataset is one YAML file. Load it into a `DataConfig`:

In [ ]:
cfg = DataConfig.from_yaml("../configs/data/busi.yaml")
cfg

**Main options** (edit the YAML, or set them in code):

- `name` — which dataset, e.g. `busi`, `kvasir`
- `size` — resolution to download: `128` / `256` / `512`
- `img_size` — square size after resize + pad
- `normalize` — `None` (/255) or `"imagenet"`
- `aug_preset` — `"medseg_default"` (augment) or `"none"`
- `n_train` — train on only N images (`None` = all)
- `batch_size`, `num_workers`, `seed`

You can also build one directly instead of from YAML:

In [ ]:
DataConfig(name="kvasir", img_size=512, aug_preset="none", batch_size=4)

## 2. Available datasets

`list_datasets()` lists every dataset as `(class, task, channels, #classes)`. Any short name, class name, or flag works for `name`

In [ ]:
list_datasets()[20:40]

In [ ]:
resolve_dataset_class("busi")   # -> BusiMSBench

## 3. Build the loaders

`build_dataloaders(cfg)` returns `{train, val, test}`. The first call downloads the dataset into `cfg.root` (when `download: true`)...

In [ ]:
loaders = build_dataloaders(cfg)
{split: len(dl.dataset) for split, dl in loaders.items()}

## 4. A batch

Every dataset yields the same dict. Images are `(B, 3, H, W)` floats; binary masks are `(B, 1, H, W)` in {0, 1}

In [ ]:
batch = next(iter(loaders["train"]))
{k: (tuple(v.shape) if hasattr(v, "shape") else v) for k, v in batch.items()}

## 5. View one sample

In [ ]:
import matplotlib.pyplot as plt

idx = 4

img = batch["image"][idx].permute(1, 2, 0).clip(0, 1)   # (H, W, 3)
msk = batch["mask"][idx, 0]                          # (H, W)

fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(img);             ax[0].set_title("image"); ax[0].axis("off")
ax[1].imshow(msk, cmap="gray"); ax[1].set_title("mask");  ax[1].axis("off")
plt.tight_layout()